# Occam 2: Turkish thinking SFT

80 training cases / 20 held-out tests. Both reasoning and final answers supervised. Use an L4/A100 or another BF16-capable CUDA GPU. This notebook has not been executed on a GPU. No accuracy result is claimed. Do not use held-out baseline mistakes to edit the data or recipe.


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select a BF16-capable GPU; T4 is not supported by this recipe."
print(torch.__version__, torch.cuda.get_device_name())


In [ ]:
%pip install transformers==5.20.2 peft==0.21.2 accelerate==1.15.0


Upload the source/data ZIP created locally by `python make_colab_bundle.py --output /EXTERNAL/occam2-colab.zip`. No credentials or weights belong in it. Restart the runtime if installation requests it, then continue.


In [ ]:
from google.colab import files, drive
from pathlib import Path
import zipfile
uploaded = files.upload()
assert len(uploaded) == 1
bundle = next(iter(uploaded))
workspace = Path("/content/occam2-code")
workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        assert (workspace / member.filename).resolve().is_relative_to(workspace.resolve())
    archive.extractall(workspace)
drive.mount("/content/drive")
artifacts = Path("/content/drive/MyDrive/llm-bender-artifacts/occam2-run-v1")
assert not artifacts.exists(), "Choose a fresh run folder; never overwrite evidence."
artifacts.mkdir(parents=True)
%cd /content/occam2-code


In [ ]:
import subprocess, sys
def run(script, *args):
    subprocess.run([sys.executable, script, *map(str,args)], check=True)
(artifacts / "requirements.lock.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
(artifacts / "nvidia-smi.txt").write_text(subprocess.check_output(["nvidia-smi"],text=True))
run("pipeline.py", "prepare", "--output", artifacts/"prepared", "--max-length", 1024)
subprocess.run([sys.executable,"-m","unittest","discover","-s",".","-p","test_pipeline.py","-v"],check=True)


Inspect `prepared/label-previews.json` and the representation report. Then run development probes. Decide on infrastructure fixes before held-out evaluation; do not tune against the held-out20.


In [ ]:
run("evaluate.py","--split","dev","--output",artifacts/"dev-base")


Freeze the protocol and dataset here. The baseline below must be preserved before training. Avoid inspecting test mistakes to revise training data.


In [ ]:
run("evaluate.py","--split","test","--output",artifacts/"base")
run("evaluate.py","--split","regression","--output",artifacts/"base-regression")


In [ ]:
run("train.py","--mode","smoke","--prepared",artifacts/"prepared","--output",artifacts/"smoke")
run("train.py","--mode","tiny","--prepared",artifacts/"prepared","--output",artifacts/"tiny")


Inspect diagnostic losses, learning pass, and saved reload generation on a training prompt. Stop if the output is broken, repetitive, or otherwise inconsistent with the targets. Full training checks manifests, but cannot replace this semantic inspection.


In [ ]:
run("train.py","--mode","full","--prepared",artifacts/"prepared",
    "--baseline",artifacts/"base/evaluation-manifest.json",
    "--smoke-evidence",artifacts/"smoke/run-manifest.json",
    "--tiny-evidence",artifacts/"tiny/run-manifest.json",
    "--output",artifacts/"sft")


In [ ]:
run("evaluate.py","--split","test","--adapter",artifacts/"sft/adapter","--output",artifacts/"candidate")
run("evaluate.py","--split","regression","--adapter",artifacts/"sft/adapter","--output",artifacts/"candidate-regression")


Score all20 pairs using `test-keys.jsonl` and RUN.md; report paired wins/losses, correctness, failures, Turkish reasoning adherence and cost. Preserve the full Drive folder before releasing the GPU. Negative or ceiling results are valid; do not replace failed questions. Export/GGUF verification is a later step if the pilot is useful.
